# Exploración computacional del triqui infinito

Este cuaderno reproduce los resultados computacionales utilizados en el artículo _El triqui infinito_. El objetivo es construir el espacio de estados del juego y verificar las propiedades mencionadas en el texto.

**Nota:** Este cuaderno no pretende implementar un jugador para el triqui infinito. Su único propósito es construir el espacio de estados del juego y verificar las propiedades estructurales utilizadas en el artículo, en particular el número de configuraciones alcanzables y la conexidad del subgrafo de estados no terminales.

#### 1. Definición del juego

In [2]:
from collections import defaultdict

# Representación de los contenidos de un escaque
EMPTY = 0
X = 1
O = 2

# Índices de las ocho líneas que forman tres en raya
WIN_LINES = [
    (0,1,2),
    (3,4,5),
    (6,7,8),
    (0,3,6),
    (1,4,7),
    (2,5,8),
    (0,4,8),
    (2,4,6)
]

In [3]:
# Tablero vacío
def empty_board():
    return (EMPTY,) * 9


def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None


# El turno se determina a partir del número de fichas de cada jugador.
def current_player(board):
    x = board.count(X)
    o = board.count(O)

    return X if x == o else O


# Genera los movimientos de la fase inicial (colocación de fichas).
def successors(state):
    board, player = state

    if winner(board) is not None:
        return []

    x = board.count(X)
    o = board.count(O)

    # La fase inicial termina cuando ya hay seis fichas en el tablero.
    if x + o == 6:
        return []

    children = []

    for i in range(9):
        if board[i] == EMPTY:
            new_board = list(board)
            new_board[i] = player

            next_player = O if player == X else X

            children.append((tuple(new_board), next_player))

    return children


def print_board(board):
    symbols = {EMPTY: ".", X: "X", O: "O"}
    for i in range(0, 9, 3):
        print(" ".join(symbols[x] for x in board[i:i+3]))
    print()

#### 2. Exploración de la fase inicial
Se exploran todas las configuraciones alcanzables durante los seis primeros turnos del juego, correspondientes a la fase en que los jugadores colocan sus fichas sobre el tablero.

In [4]:
# Grafo de estados construido durante la exploración
graph = defaultdict(set)
visited = set()

# Recorre recursivamente todas las configuraciones alcanzables
# durante la fase inicial del juego.
def dfs(state):
    if state in visited:
        return

    visited.add(state)

    for child in successors(state):
        graph[state].add(child)
        dfs(child)


# Exploración completa de la fase de colocación
initial_state = (empty_board(), X)
dfs(initial_state)

# Configuraciones alcanzables al terminar el sexto turno
six_move_positions = [
    state for state in visited
    if state[0].count(X) + state[0].count(O) == 6
]

print("Número de configuraciones alcanzables hasta el sexto turno:", len(visited))
print("Número de estados alcanzables al finalizar el sexto turno:", len(six_move_positions))

Número de configuraciones alcanzables hasta el sexto turno: 3870
Número de estados alcanzables al finalizar el sexto turno: 1520


#### 3. Exploración de la fase de movimiento
A partir del sexto turno el número de fichas permanece constante. Desde cada una de las configuraciones alcanzables se generan todos los movimientos legales de esta segunda fase.

In [13]:
# Estados alcanzables al terminar la fase inicial
six_move_positions = set(six_move_positions)

def successors_phase2(state):
    """Genera todos los movimientos legales una vez las seis
    fichas ya están sobre el tablero.
    En cada turno el jugador mueve una de sus tres fichas
    a cualquiera de los tres espacios vacíos.
    """
    board, player = state

    if winner(board):
        return []

    children = []

    player_positions = [i for i, x in enumerate(board) if x == player]
    empty_positions = [i for i, x in enumerate(board) if x == EMPTY]

    for origin in player_positions:
        for destination in empty_positions:
            new_board = list(board)

            new_board[origin] = EMPTY
            new_board[destination] = player

            next_player = O if player == X else X

            children.append((tuple(new_board), next_player))

    return children


# Grafo de posiciones a partir del sexto turno.
# El jugador que mueve se conserva para la exploración,
# pero el grafo se construye únicamente sobre tableros.
phase2_graph = defaultdict(set)

visited_states = set(six_move_positions)
stack = list(six_move_positions)

reachable_boards = {board for board, _ in six_move_positions}

while stack:
    state = stack.pop()

    for new_state in successors_phase2(state):
        new_board, _ = new_state

        # Los estados terminales no tienen sucesores.
        if winner(new_board):
            continue

        reachable_boards.add(new_board)

        board, _ = state
        phase2_graph[board].add(new_board)

        if new_state not in visited_states:
            visited_states.add(new_state)
            stack.append(new_state)

print("Estados explorados:", len(visited_states))
print("Tableros alcanzables:", len(reachable_boards))

Estados explorados: 2892
Tableros alcanzables: 1520


#### 4. Identificación de estados terminales
Clasificamos los tableros alcanzables según correspondan a posiciones terminales o no terminales.

In [20]:
# Clasificamos los tableros alcanzables según sean terminales o no.
terminal_boards = set()
nonterminal_boards = set()

for board in reachable_boards:
    if winner(board):
        terminal_boards.add(board)
    else:
        nonterminal_boards.add(board)

print("Tableros no terminales:", len(nonterminal_boards))
print("Tableros terminales:", len(terminal_boards))

Tableros no terminales: 1372
Tableros terminales: 148


#### 5. Verificación de la conexidad
Finalmente verificamos que el subgrafo inducido por los estados no terminales es fuertemente conexo.

In [21]:
# Verificamos si el subgrafo inducido por los estados no terminales
# es fuertemente conexo.
def strongly_connected(graph, vertices):
    """Comprueba si el grafo dirigido inducido por 'vertices'
    es fuertemente conexo."""

    start = next(iter(vertices))

    def dfs(g):
        visited = set()
        stack = [start]

        while stack:
            v = stack.pop()

            if v in visited:
                continue

            visited.add(v)
            stack.extend(g[v])

        return visited

    # Todo vértice debe ser alcanzable desde uno cualquiera.
    if dfs(graph) != vertices:
        return False

    # Construimos el grafo con todas las aristas invertidas.
    reverse = defaultdict(set)

    for v in vertices:
        reverse[v]      # Fuerza la creación del vértice.

    for u in vertices:
        for v in graph[u]:
            reverse[v].add(u)

    # En el grafo transpuesto también debe alcanzarse todo vértice.
    return dfs(reverse) == vertices


print(strongly_connected(phase2_graph, nonterminal_boards))

True


#### Resultados
- Configuraciones alcanzables al finalizar el sexto turno: 1520
- Estados terminales: 148
- Estados no terminales: 1372
- El subgrafo inducido por los estados no terminales es fuertemente conexo.